# 부품 자동추천 AI — Colab 파이프라인 (수집 + QLoRA 학습)

Google Colab **Pro (A100 40GB 권장)** 에서 데이터 수집부터 Qwen3-14B QLoRA 학습·추론까지 한 번에 실행합니다.

1. GPU/환경 확인 → 2. Google Drive 마운트 → 3. 코드/의존성 설치 → 4. (수집용) Ollama 기동 →
5. 시크릿 설정 → 6. 데이터 수집 → 7. 전처리 → 8. QLoRA 학습 → 9. 추론 테스트

> **런타임 → 런타임 유형 변경**에서 A100을 선택했는지 먼저 확인하세요. Colab 런타임은 휘발성이므로 데이터/어댑터는 모두 Google Drive에 저장합니다.

## 1. GPU 확인

In [ ]:
!nvidia-smi

## 2. Google Drive 마운트
수집 데이터·학습 산출물(어댑터)을 Drive에 영구 저장합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/auto_search'   # Drive 내 저장 위치
os.makedirs(f'{DRIVE_ROOT}/datasheets', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/models', exist_ok=True)
print('Drive 저장 경로:', DRIVE_ROOT)

## 3. 코드 가져오기 + 의존성 설치
아래 `REPO_URL`을 본인 저장소 주소로 바꾸세요. (또는 Drive에 올려둔 코드를 사용)

In [ ]:
REPO_URL = 'https://github.com/Merlion1346/Auto_Part_Search.git'
%cd /content
![ -d Auto_Part_Search ] || git clone $REPO_URL
%cd /content/Auto_Part_Search

# 데이터시트 PDF 파싱용 Java (OpenDataLoader 는 JVM 기반)
!apt-get -qq install -y default-jre > /dev/null && java -version

# Python 의존성: Colab 기본 torch/transformers는 그대로 두고 부족한 것만 설치/갱신
# (requirements.txt 는 torch를 핀하지 않고 최소 버전만 지정)
!pip install -q -U -r requirements.txt

## 4. (데이터 수집용) llama.cpp 서버 기동
수집 단계의 사양 추출/요구사항 증강은 OpenAI 호환 LLM 엔드포인트를 사용합니다.
llama.cpp 를 CUDA로 빌드한 뒤 `llama-server` 로 Qwen3-14B(GGUF)를 OpenAI 호환(`/v1`)으로 서빙합니다.

> `response_format=json_object` 가 grammar 로 JSON 출력을 강제하므로, Qwen3 thinking 토큰이 섞여도 파싱이 안전합니다.
>
> 이미 만들어 둔 카탈로그(`parts_catalog.json`)를 Drive에서 재사용한다면 이 셀과 6번(수집)은 건너뛰어도 됩니다. 외부 API(DashScope 등)를 쓸 경우엔 5번에서 `LLM_*`만 바꾸고 이 셀을 건너뜁니다.

In [ ]:
# llama.cpp 빌드 (CUDA). -hf 자동 다운로드를 위해 libcurl 필요
!apt-get -qq install -y libcurl4-openssl-dev > /dev/null
![ -d /content/llama.cpp ] || git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=ON -DLLAMA_CURL=ON
!cmake --build /content/llama.cpp/build --config Release -j --target llama-server

import subprocess, time, urllib.request
# Qwen3-14B GGUF(Q4_K_M)를 HF에서 자동 다운로드하며 OpenAI 호환 서버 기동
server = subprocess.Popen([
    "/content/llama.cpp/build/bin/llama-server",
    "-hf", "Qwen/Qwen3-14B-GGUF:Q4_K_M",
    "--host", "127.0.0.1", "--port", "8000",
    "-ngl", "999", "-c", "8192", "--jinja",
])

# 첫 실행은 GGUF(~9GB) 다운로드 때문에 시간이 걸릴 수 있다 (최대 ~10분 대기)
for _ in range(300):
    try:
        urllib.request.urlopen("http://localhost:8000/health", timeout=2)
        print("llama-server 준비됨"); break
    except Exception:
        time.sleep(2)
else:
    print("경고: 서버가 아직 응답하지 않습니다(모델 다운로드 지연 가능). 잠시 후 다음 셀을 실행하세요.")

## 5. 시크릿 / .env 설정
Colab 좌측 🔑(보안 비밀)에 `MOUSER_API_KEY`(또는 Digi-Key 키)를 등록한 뒤 실행하세요.
경로는 Drive로 지정해 런타임 재시작에도 보존되게 합니다.

In [ ]:
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name) or ''
    except Exception:
        return ''

env = f"""
MOUSER_API_KEY={secret('MOUSER_API_KEY')}
MOUSER_BASE=https://api.mouser.com
DIGIKEY_CLIENT_ID={secret('DIGIKEY_CLIENT_ID')}
DIGIKEY_CLIENT_SECRET={secret('DIGIKEY_CLIENT_SECRET')}
DIGIKEY_BASE=https://api.digikey.com

LLM_BASE_URL=http://localhost:8000/v1
LLM_API_KEY=EMPTY
LLM_MODEL=qwen3-14b

PDF_DIR={DRIVE_ROOT}/datasheets
CATALOG_PATH={DRIVE_ROOT}/parts_catalog.json
"""
with open('.env', 'w') as f:
    f.write(env)
print('.env 작성 완료')

## 6. 데이터 수집 (유통사 API + 데이터시트 파싱 + LLM 사양 추출)
→ `parts_catalog.json` 생성(Drive에 저장)

In [ ]:
!python -m pipeline.build_catalog --source mouser \
    --keywords-file pipeline/keywords.example.txt --limit 15

## 7. 학습 데이터 전처리
카탈로그 → TRL 대화 데이터셋(`train.jsonl`). `--augment` 는 Ollama로 요구사항을 다양화합니다.

In [ ]:
!python train/preprocess.py \
    --input-glob "$DRIVE_ROOT/parts_catalog.json" \
    --output "$DRIVE_ROOT/train.jsonl" \
    --augment --num-variations 5

## 8. QLoRA 학습 (Qwen3-14B)
어댑터는 Drive에 저장됩니다. A100 40GB 기준 기본값입니다. VRAM이 작은 GPU(L4/T4)면 `--max-seq-len 1024 --batch-size 1` 로 낮추세요.

In [ ]:
!python train/train_qlora.py \
    --train-file "$DRIVE_ROOT/train.jsonl" \
    --output-dir "$DRIVE_ROOT/models/qwen3-14b-parts-lora"

## 9. 추론 테스트

In [ ]:
!python src/recommend.py \
    --adapter "$DRIVE_ROOT/models/qwen3-14b-parts-lora" \
    "5V를 3.3V로 변환하고 1A 이상 출력하는 LDO 추천해줘"